In [0]:
from pyspark.sql import functions as F

CATALOG = "samplework"

BASE_PATH = "abfss://ecommerce@anils.dfs.core.windows.net/source/"

VOLUME_PATH = "/Volumes/samplework/bronze/ecommerce_volume"

SCHEMA_PATH = f"{VOLUME_PATH}/schemas"

CHECKPOINT_ROOT = f"{VOLUME_PATH}/checkpoints/bronze"

In [0]:
customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_PATH}/customers"
        )
        .load(f"{BASE_PATH}customers/")
)


In [0]:
customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_PATH}/customers"
        )
        .load(f"{BASE_PATH}customers/")
)

In [0]:

customers_bronze = (
    customers_stream
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)


In [0]:
%sql
SHOW EXTERNAL LOCATIONS;

In [0]:
%sql
DESCRIBE EXTERNAL LOCATION ecommerce_external_location;

In [0]:
dbutils.fs.ls(
    "abfss://ecommerce@anils.dfs.core.windows.net/source/customers/"
)

In [0]:
customers_query = (
    customers_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{CHECKPOINT_ROOT}/customers"
        )
        .trigger(availableNow=True)
        .toTable("samplework.bronze.customers")
)

In [0]:
%sql
SELECT *
FROM samplework.bronze.customers;

In [0]:


# ============================================================
# 1. CUSTOMERS - BRONZE
# ============================================================




customers_query = (
    customers_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{CHECKPOINT_ROOT}/customers"
        )
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.bronze.customers")
)


# ============================================================
# 2. PRODUCTS - BRONZE
# ============================================================

products_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_PATH}/products"
        )
        .load(f"{BASE_PATH}products/")
)

products_bronze = (
    products_stream
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

products_query = (
    products_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{CHECKPOINT_ROOT}/products"
        )
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.bronze.products")
)

In [0]:



# ============================================================
# 3. ORDERS - BRONZE
# ============================================================

orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option(
            "cloudFiles.schemaLocation",
            f"{SCHEMA_PATH}/orders"
        )
        .load(f"{BASE_PATH}orders/")
)

orders_bronze = (
    orders_stream
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file",
            F.col("_metadata.file_path")
        )
)

orders_query = (
    orders_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            f"{CHECKPOINT_ROOT}/orders"
        )
        .trigger(availableNow=True)
        .toTable(f"{CATALOG}.bronze.orders")
)


print("Bronze Auto Loader pipelines started successfully.")
